# YUKTHI 2026 — Chiller Intelligence Training

Trains the CatBoost expected-energy model with chronological validation.

**Pipeline:** Load → Preprocess → Feature Engineer → Train (TimeSeriesSplit) → Evaluate → Save

In [ ]:
# Cell 1: Install dependencies
!pip install -q catboost shap pandas numpy scikit-learn matplotlib plotly

In [ ]:
# Cell 2: Upload or mount data
import os
import sys

# Option A: Upload from local machine
from google.colab import files
# uploaded = files.upload()  # Uncomment to upload CSV

# Option B: Clone from GitHub
if not os.path.exists('yukthi'):
    !git clone https://github.com/YOUR_USERNAME/yukthi.git

os.chdir('yukthi')
sys.path.insert(0, '.')
print(f'Working directory: {os.getcwd()}')
print(f'Files: {os.listdir(".")}')

In [ ]:
# Cell 3: Import pipeline modules
import logging
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.data_loader import load_and_validate
from src.preprocessing import preprocess
from src.features import engineer_features
from src.train import train_model
from src.predict import predict_expected_energy
from src.anomaly import score_anomalies
from src.explain import explain_prediction, generate_narrative
from src.constants import FEATURE_COLS, COL_ENERGY, COL_TIMESTAMP, COL_EQUIPMENT_ID

logging.basicConfig(level=logging.INFO)
print('All modules imported successfully.')

In [ ]:
# Cell 4: Load and validate data
csv_path = Path('data/raw/chiller_data.csv')
df = load_and_validate(csv_path)
print(f'Loaded: {df.shape}')
print(f'Equipment: {df[COL_EQUIPMENT_ID].unique().tolist()}')
print(f'Date range: {df[COL_TIMESTAMP].min()} to {df[COL_TIMESTAMP].max()}')

In [ ]:
# Cell 5: Preprocess
df, gap_report = preprocess(df)
print(f'After preprocessing: {df.shape}')
print(f'Temporal gaps per equipment: {gap_report.total_gaps_per_equipment}')
print(f'Remaining NaN: {df.isna().sum().sum()}')

In [ ]:
# Cell 6: Feature engineering
df = engineer_features(df)
print(f'Features added. Shape: {df.shape}')
print(f'Feature columns: {FEATURE_COLS}')

In [ ]:
# Cell 7: Train model with chronological validation
model_path = Path('models/catboost_expected_energy.cbm')
model_path.parent.mkdir(parents=True, exist_ok=True)

# Use GPU if available in Colab
import subprocess
try:
    subprocess.check_output(['nvidia-smi'])
    task_type = 'GPU'
    print('GPU detected — training on GPU')
except (subprocess.CalledProcessError, FileNotFoundError):
    task_type = 'CPU'
    print('No GPU — training on CPU')

result = train_model(
    df, FEATURE_COLS, COL_ENERGY, model_path,
    task_type=task_type,
)
print(result.summary())

In [ ]:
# Cell 8: Generate predictions and residuals
df = predict_expected_energy(result.model, df, FEATURE_COLS)
print(f'Predictions generated. Mean residual: {df["residual"].mean():.2f} kWh')

In [ ]:
# Cell 9: Anomaly scoring
df, events = score_anomalies(df)
print(f'Anomaly events: {len(events)}')
for event in events[:5]:
    print(event.describe())
    print()

In [ ]:
# Cell 10: Visualize actual vs expected energy
fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True)

for i, eid in enumerate(sorted(df[COL_EQUIPMENT_ID].unique())):
    edf = df[df[COL_EQUIPMENT_ID] == eid].sort_values(COL_TIMESTAMP)
    ax = axes[i]
    ax.plot(edf[COL_TIMESTAMP], edf[COL_ENERGY], label='Actual', alpha=0.7, linewidth=0.8)
    ax.plot(edf[COL_TIMESTAMP], edf['expected_energy'], label='Expected', alpha=0.7, linewidth=0.8, linestyle='--')
    
    anomalies = edf[edf['anomaly_flag'] == 1]
    if not anomalies.empty:
        ax.scatter(anomalies[COL_TIMESTAMP], anomalies[COL_ENERGY], 
                   color='red', s=10, label='Anomaly', zorder=5)
    
    ax.set_title(eid)
    ax.set_ylabel('Energy (kWh)')
    ax.legend(loc='upper right')

plt.tight_layout()
plt.savefig('actual_vs_expected.png', dpi=150)
plt.show()

In [ ]:
# Cell 11: SHAP explanation for sample anomaly
anomaly_rows = df[df['anomaly_flag'] == 1]
if not anomaly_rows.empty:
    sample = anomaly_rows.iloc[[0]]
    explanation = explain_prediction(result.model, sample, FEATURE_COLS)
    narrative = generate_narrative(explanation)
    print(f'Timestamp: {sample[COL_TIMESTAMP].iloc[0]}')
    print(f'Equipment: {sample[COL_EQUIPMENT_ID].iloc[0]}')
    print(f'Actual: {sample[COL_ENERGY].iloc[0]:.1f} kWh')
    print(f'Expected: {explanation.predicted_value:.1f} kWh')
    print(f'\nExplanation: {narrative}')
    print(f'\nTop contributions:')
    for c in explanation.top_contributors:
        print(f'  {c.feature_name}: {c.shap_value:+.2f} kWh ({c.direction})')
else:
    print('No anomalies detected in this run.')

In [ ]:
# Cell 12: Download trained model
from google.colab import files
files.download('models/catboost_expected_energy.cbm')
print('Model downloaded.')